# v17: class imbalance on image and video, with TransCLIP and ZLaP

This notebook repeats Table 2's imbalance protocol on the image and video bundles you built in v8/v9 (`cache/<dataset>_<encoder>.pt`): **ImageNet-1k, Places365, Kinetics-400 (3,998-clip subset)** × **ImageBind, LanguageBind**.

* **Methods.** cosine, WO+P, EM-WO+P (ours), TransCLIP and ZLaP. TransCLIP and ZLaP use their authors' code at the same commits and settings as v16.
* **Conditions.** Balanced, plus 10× and 50× imbalance, seeds 0, 1, 2 (identical subsampler to the paper).
* **Anchors.** The papers' templates only. No LLM descriptions exist for these class sets.
* **Memory.** To fit TransCLIP on a T4 for 50k images × 1000 classes, two changes affect memory only: chunk sizes, and the 3-NN graph computed in blocks. Both were checked to give identical predictions on ESC-50 and US8K.
* **Check.** The balanced cosine and WO+P numbers are compared with Table 1, to confirm these are the same bundles.

**Run:** Runtime → **T4 GPU** (an A100 is faster but not needed) → Run all.
* It takes roughly 2–3 hours, mostly ImageNet.
* It saves after every single subsample, so if Colab disconnects, just Run all again and it continues where it stopped.
* Kinetics finishes first (about 10 minutes); its numbers are useful on their own.

**Send back:** `MyDrive/ortho-v7/results/v17_imbalance_vision.csv`. If convenient, also send the `v17_*.json` files from the same folder.

In [ ]:
import os, sys, json, glob, subprocess
from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/ortho-v7"; os.makedirs(f"{BASE}/results", exist_ok=True); os.chdir(BASE)
NAMES = {"imagenet": ["imagenet"], "places365": ["places365"], "kinetics400": ["kinetics400", "k400"]}
RUNS = []
for ds in ["kinetics400", "places365", "imagenet"]:                    # small first, ImageNet last
    for enc in ["imagebind", "languagebind"]:
        cands = [f"cache/{n}_{enc}.pt" for n in NAMES[ds]]
        hits = [c for c in cands if os.path.exists(c)] or [p for n in NAMES[ds] for p in glob.glob(f"/content/drive/MyDrive/**/{n}_{enc}.pt", recursive=True)]
        print(f"{ds:12s} {enc:12s} -> {hits[0] if hits else 'NOT FOUND (skipped)'}")
        if hits: RUNS.append({"ds": ds, "enc": enc, "bundle": hits[0]})

In [ ]:
# authors' code at the v16 commits; TransCLIP's fixed chunk size becomes a variable (memory only)
for url, d, sha in [("https://github.com/MaxZanella/transduction-for-vlms.git", "/content/_TransCLIP", "6a9e8f3ba57a11d1c5affe248a82df56dbc7f1b8"),
                    ("https://github.com/vladan-stojnic/ZLaP.git", "/content/_ZLaP", "89ab14bef8904a9397f8b6a7e5f3e6af2c288b93")]:
    if not os.path.exists(d): subprocess.check_call(["git", "clone", "-q", url, d])
    subprocess.check_call(["git", "-C", d, "checkout", "-q", sha])
f = "/content/_TransCLIP/TransCLIP_solver/TransCLIP_utils.py"; src = open(f).read()
if not src.startswith("CHUNK = 2500"):
    assert src.count("chunk_size = 2500") == 2, "TransCLIP code changed"
    open(f, "w").write("CHUNK = 2500  # patched: chunk size set per dataset (memory only)\n" + src.replace("chunk_size = 2500", "chunk_size = CHUNK"))
try:
    import cupy; print("cupy", cupy.__version__)
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"]); import cupy; print("cupy", cupy.__version__)
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")
import psutil; print(f"RAM: {psutil.virtual_memory().total / 2**30:.0f} GB")

In [ ]:
%%writefile llm_lib.py
"""llm_lib.py - label-free zero-shot methods that start from ANY initial score matrix S0 (N x C), so that
LLM-generated prototypes and the covariance method can be combined and compared on equal terms.

Same frozen settings as the main study (geo_final.py / ortho_lib.py): eps = 0.02, gamma = 0.1, k = 10, alpha = 0.5.
No function here reads labels.
"""
import numpy as np

GAMMA, EPS, K_NN, ALPHA, TAU_DECIDE, TAU_ZS = 0.1, 0.02, 10, 0.5, 0.1, 0.01

def unit(X, eps=1e-12):
    return X / np.maximum(np.linalg.norm(X, axis=-1, keepdims=True), eps)

def _softmax(L):
    L = L - L.max(1, keepdims=True); P = np.exp(L); return P / P.sum(1, keepdims=True)

# ------------------------------------------------------------------ mass geometry (pseudo-labels)
def sinkhorn(S, eps=EPS, it=100):
    """Balanced soft assignment: every class gets N/C inputs."""
    L = S / eps; L = L - L.max(1, keepdims=True); Q = np.exp(L); N, C = S.shape
    for _ in range(it):
        Q /= Q.sum(1, keepdims=True); Q *= (N / C) / Q.sum(0)
    return Q / Q.sum(1, keepdims=True)

def sinkhorn_marg(S, eps, pi, it=100):
    L = S / eps; L = L - L.max(1, keepdims=True); Q = np.exp(L); N = len(S)
    for _ in range(it):
        Q /= Q.sum(1, keepdims=True); Q *= (N * pi) / np.maximum(Q.sum(0), 1e-300)
    return Q / Q.sum(1, keepdims=True)

def ot_unbalanced(S, eps=EPS, rho=EPS, iters=200):
    N, C = S.shape; L = S / eps; L = L - L.max(1, keepdims=True); K = np.exp(L)
    a = np.ones(N); b = np.ones(C); f = rho / (rho + eps); target = N / C
    for _ in range(iters):
        a = 1.0 / np.maximum(K @ b, 1e-300)
        b = (target / np.maximum(K.T @ a, 1e-300)) ** f
    Q = a[:, None] * K * b[None, :]
    return Q / Q.sum(1, keepdims=True)

def em_prior(L, iters=100, floor=1e-4):
    N, C = L.shape; pi = np.full(C, 1.0 / C)
    for _ in range(iters):
        new = np.maximum(_softmax(L + np.log(pi)).mean(0), floor); new /= new.sum()
        if np.abs(new - pi).max() < 1e-6: pi = new; break
        pi = new
    return pi, _softmax(L + np.log(pi))

# ------------------------------------------------------------------ noise geometry (covariance score)
def wo_fit_soft(Z, Q, gamma=GAMMA):
    """Soft class means and shrunk within-class covariance from a soft assignment Q; returns (M, Sigma^-1)."""
    N, D = Z.shape; w = Q.sum(0) + 1e-9; M = (Q.T @ Z) / w[:, None]
    Sw = (Z.T @ Z - (M.T * w) @ M) / N; Sw = (1 - gamma) * Sw + gamma * np.trace(Sw) / D * np.eye(D)
    return M, np.linalg.inv(Sw)

def wo_scores(Z, M, Wi):
    MW = M @ Wi
    return Z @ MW.T - 0.5 * np.sum(MW * M, 1)

def whiten_op(Wi):
    lam, U = np.linalg.eigh(np.linalg.inv(Wi))
    return (U / np.sqrt(lam)) @ U.T

# ------------------------------------------------------------------ data geometry (k-NN smoothing)
def knn_graph(X, k=K_NN, chunk=2000):
    N = len(X); idx = np.empty((N, k), int); w = np.empty((N, k))
    for s in range(0, N, chunk):
        G = X[s:s + chunk] @ X.T
        G[np.arange(G.shape[0]), np.arange(s, s + G.shape[0])] = -np.inf
        ii = np.argpartition(-G, k, axis=1)[:, :k]; idx[s:s + chunk] = ii; w[s:s + chunk] = np.take_along_axis(G, ii, 1)
    return idx, w

def _prop(w, Q, idx):
    out = np.zeros_like(Q)
    for j in range(idx.shape[1]): out += w[:, j:j + 1] * Q[idx[:, j]]
    return out

def propagate(Z, Q0, Wi, k=K_NN, alpha=ALPHA, iters=10, power=3):
    X = unit((Z - Z.mean(0)) @ whiten_op(Wi))
    idx, w = knn_graph(X, min(k, len(Z) - 1)); w = np.maximum(w, 0) ** power; w /= w.sum(1, keepdims=True) + 1e-12
    Q = Q0.copy()
    for _ in range(iters): Q = (1 - alpha) * Q0 + alpha * _prop(w, Q, idx)
    return Q

# ------------------------------------------------------------------ the three label-free methods, from any S0
def wop(Z, S0):
    """WO+P: equal class shares (the main method in the talk)."""
    M, Wi = wo_fit_soft(Z, sinkhorn(S0)); L = wo_scores(Z, M, Wi)
    return propagate(Z, sinkhorn(L / L.std(), TAU_DECIDE), Wi).argmax(1)

def pa_wop(Z, S0, smooth=1.0):
    """PA-WO+P: estimate class proportions first, then balance to them."""
    C = S0.shape[1]; M, Wi = wo_fit_soft(Z, ot_unbalanced(S0))
    pi = (np.bincount(wo_scores(Z, M, Wi).argmax(1), minlength=C) + smooth) / (len(Z) + smooth * C)
    M, Wi = wo_fit_soft(Z, sinkhorn_marg(S0, EPS, pi)); L = wo_scores(Z, M, Wi)
    return propagate(Z, _softmax(L / L.std() / TAU_DECIDE), Wi).argmax(1)

def em_wop(Z, S0):
    """EM-WO+P: never assumes balance (EM-estimated prior on the initial scores)."""
    M, Wi = wo_fit_soft(Z, em_prior(S0 / TAU_ZS)[1]); L = wo_scores(Z, M, Wi)
    return propagate(Z, _softmax(L / L.std() / TAU_DECIDE), Wi).argmax(1)

METHODS = {"WO+P": wop, "PA-WO+P": pa_wop, "EM-WO+P": em_wop}

# ------------------------------------------------------------------ anchors: templates, LLM agents, combinations
def build_anchor_scores(Z, T_tmpl, descs, topk=10):
    """T_tmpl: C x D template prototypes (the papers' protocol). descs: {source: [C arrays of (n_c x D) unit vectors]}.
    Returns {anchor name: S (N x C)}."""
    S = {"templates": Z @ T_tmpl.T}
    means = {}
    for src, per_class in descs.items():
        T = unit(np.stack([unit(D.mean(0)) if len(D) else T_tmpl[c] for c, D in enumerate(per_class)]))
        means[src] = T; S[f"LLM:{src}"] = Z @ T.T
    if len(descs) > 1:
        allD = [np.concatenate([descs[s][c] for s in descs if len(descs[s][c])] or [T_tmpl[c:c + 1]]) for c in range(len(T_tmpl))]
        T_all = unit(np.stack([unit(D.mean(0)) for D in allD])); S["LLM:all agents"] = Z @ T_all.T
    else:
        allD = list(next(iter(descs.values()))); T_all = next(iter(means.values()))
    S["templates + LLM"] = Z @ unit(T_tmpl + T_all).T
    # UniBind-style: per class keep the top-k descriptions closest to the template prototype, score = max over them
    Smax = np.empty((len(Z), len(T_tmpl)))
    for c, D in enumerate(allD):
        D = unit(D); keep = D[np.argsort(-(D @ T_tmpl[c]))[:topk]] if len(D) else T_tmpl[c:c + 1]
        Smax[:, c] = (Z @ keep.T).max(1)
    S["UniBind-style max"] = Smax
    return S

# ------------------------------------------------------------------ evaluation helpers
def skew_idx(y, ir, seed):
    rng = np.random.default_rng(seed); C = y.max() + 1; n0 = np.bincount(y, minlength=C).min(); order = rng.permutation(C); keep = []
    for r, c in enumerate(order):
        ix = rng.permutation(np.flatnonzero(y == c)); keep.append(ix[:max(1, int(round(n0 * ir ** (-r / (C - 1)))))])
    return np.sort(np.concatenate(keep))

def mcnemar(ok_a, ok_b):
    from scipy.stats import binomtest
    fixed = int(np.sum(~ok_a & ok_b)); broken = int(np.sum(ok_a & ~ok_b))
    p = binomtest(fixed, fixed + broken, 0.5).pvalue if fixed + broken else 1.0
    return fixed, broken, float(p)


In [ ]:
%%writefile v16_baselines.py
"""v16_baselines.py - TransCLIP (NeurIPS'24) and ZLaP (CVPR'24) on the SAME cached ImageBind / LanguageBind embeddings,
anchors, subsamples and metric as our Table 2, so the numbers drop straight into the paper.

* TransCLIP: the authors' own solver (github.com/MaxZanella/transduction-for-vlms @ 6a9e8f3), zero-shot mode, their defaults.
* ZLaP: the authors' own transductive code (github.com/vladan-stojnic/ZLaP @ 89ab14b), README setting k=5, gamma=5, alpha=0.3.
  Only their FAISS k-NN search is replaced by an exact torch top-k (same inner-product neighbours), because faiss-gpu does not
  pip-install on Colab. A line-by-line scipy port (zlap_port) is also run on ESC-50 as a cross-check.
* Ours (cosine, WO+P, EM-WO+P) from llm_lib, as a sanity check that this notebook reproduces the paper's numbers.
Anchors: T = the papers' templates; T+D = unit(T + mean of the LLM description embeddings), exactly as in v11 / Table 2.
Conditions: balanced, IR10 and IR50 with seeds 0, 1, 2 (llm_lib.skew_idx), mean over seeds.
Labels are used only for scoring (TransCLIP's solver also reads them to count the classes and print its own accuracy)."""
import os, sys, json, time, types, importlib, numpy as np, torch
import llm_lib as L

DEV = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = (0, 1, 2); IRS = (10, 50)
ZLAP_K, ZLAP_GAMMA, ZLAP_ALPHA = 5, 5.0, 0.3
ZLAP_IMPL = None

# ------------------------------------------------------------------ data
def load_run(bundle, text_cache=None):
    d = torch.load(bundle, map_location="cpu", weights_only=False)
    Z = L.unit(d["emb"].double().numpy()); T = L.unit(d["text_emb"].double().numpy())
    y = d["labels"].numpy().astype(int); names = list(d["classnames"]); anchors = {"T": T}
    if text_cache and os.path.exists(text_cache):          # same construction as llm_lib.build_anchor_scores 'templates + LLM'
        TX = torch.load(text_cache, map_location="cpu", weights_only=False)
        pooled = []
        for c, n in enumerate(names):
            parts = [TX[s][n].double().numpy() for s in TX if n in TX[s] and len(TX[s][n])]
            pooled.append(L.unit(np.concatenate(parts)).mean(0) if parts else T[c])
        T_all = L.unit(np.stack([L.unit(p) for p in pooled])); anchors["T+D"] = L.unit(T + T_all)
    return Z, y, names, anchors

# ------------------------------------------------------------------ TransCLIP (authors' solver)
_TC = None
def _transclip_solver(repo):
    global _TC
    if _TC is None:
        if DEV == "cpu":                                   # local CPU test only: their code calls .cuda() unconditionally
            torch.Tensor.cuda = lambda self, *a, **k: self; torch.nn.Module.cuda = lambda self, *a, **k: self
        sys.path.insert(0, os.path.abspath(repo))
        mod = importlib.import_module("TransCLIP_solver.TransCLIP")
        # their cls_acc only prints accuracy; float() of a 1-element array fails on recent NumPy, so use .item()
        mod.cls_acc = lambda out, tgt, topk=1: 100.0 * (out.argmax(1) == tgt).float().mean().item()
        _TC = mod.TransCLIP_solver
    return _TC

def transclip(Z, y, T, repo="_TransCLIP"):
    solve = _transclip_solver(repo)
    q = torch.tensor(Z, dtype=torch.float32, device=DEV); lab = torch.tensor(y, device=DEV)
    P = torch.tensor(T.T.copy(), dtype=torch.float32, device=DEV)        # (d, K) as their API expects
    z, _ = solve(None, None, None, None, q, lab, P, verbose=False)
    return z.argmax(1).cpu().numpy()

# ------------------------------------------------------------------ ZLaP (authors' code, exact k-NN instead of FAISS)
_ZL = None
def _exact_knn(X, Q, k):
    """Same contract as ZLaP's search_faiss(X, Q, k): for each row of Q, the k rows of X with largest inner product."""
    Xt = torch.tensor(X, device=DEV); out_i, out_s = [], []
    for s in range(0, len(Q), 4096):
        S = torch.tensor(Q[s:s + 4096], device=DEV) @ Xt.T; v, i = S.topk(k, dim=1)
        out_i.append(i.cpu().numpy()); out_s.append(v.cpu().numpy())
    return np.concatenate(out_i).astype(np.int64), np.concatenate(out_s).astype(np.float32)

def _zlap_module(repo):
    global _ZL
    if _ZL is None:
        sys.modules.setdefault("faiss", types.ModuleType("faiss"))   # only used by search_faiss, which we replace
        sys.path.insert(0, os.path.abspath(repo))
        _ZL = importlib.import_module("zlap"); _ZL.search_faiss = _exact_knn
        import inspect, cupyx.scipy.sparse.linalg as cl
        if "tol" not in inspect.signature(cl.cg).parameters:          # newer CuPy renamed tol -> rtol
            _ZL.dfs_search = lambda Lm, Y, tol=1e-6, maxiter=50, cast_to_numpy=True: (
                (lambda o: o.get() if cast_to_numpy else o)(cl.cg(Lm, Y, rtol=tol, maxiter=maxiter)[0]))
    return _ZL

def zlap(Z, T, repo="_ZLaP", k=ZLAP_K, gamma=ZLAP_GAMMA, alpha=ZLAP_ALPHA):
    m = _zlap_module(repo)
    return m.do_transductive_lp(Z.astype(np.float32), T.astype(np.float32), k, gamma, alpha).argmax(1)

def zlap_port(Z, T, k=ZLAP_K, gamma=ZLAP_GAMMA, alpha=ZLAP_ALPHA):
    """Line-by-line scipy port of ZLaP's do_transductive_lp (used for the CPU test and as a cross-check)."""
    import scipy.sparse as sp; from scipy.sparse.linalg import cg
    X = Z.astype(np.float32); clf = T.astype(np.float32); C = len(clf)
    ki, si = _exact_knn(X, X, min(k, len(X))); kt, st = _exact_knn(clf, X, min(k, C))
    knn = np.concatenate([np.concatenate([ki + C, kt], 1)], 0); sim = np.concatenate([si, st], 1)
    knn = np.concatenate([-np.ones((C, knn.shape[1]), knn.dtype), knn]); sim = np.concatenate([np.zeros((C, sim.shape[1]), sim.dtype), sim])
    sim[sim < 0] = 0; mk = knn < C; sim[mk] = sim[mk] ** gamma
    N = len(knn); rows = np.repeat(np.arange(N), knn.shape[1]); kf = knn.ravel(); sf = sim.ravel(); v = kf != -1
    W = sp.csr_matrix((sf[v], (rows[v], kf[v])), shape=(N, N)); W = W + W.T
    W = W - sp.diags(W.diagonal(), 0); Sr = np.asarray(W.sum(1)).ravel(); Sr[Sr == 0] = 1
    Dm = sp.diags(1.0 / np.sqrt(Sr), 0); Lp = sp.eye(N) - alpha * (Dm @ W @ Dm)
    out = np.zeros((len(X), C))
    for c in range(C):
        Y = np.zeros(N); Y[c] = 1; out[:, c] = cg(Lp, Y, rtol=1e-6, atol=0.0, maxiter=50)[0][C:]
    return out.argmax(1)

# ------------------------------------------------------------------ evaluation
OURS = {"cosine": lambda Z, y, T: (Z @ T.T).argmax(1),
        "WO+P": lambda Z, y, T: L.wop(Z, Z @ T.T),
        "EM-WO+P": lambda Z, y, T: L.em_wop(Z, Z @ T.T)}

def methods(tc_repo, zl_repo, zlap_impl="official"):
    """zlap_impl='official' uses the authors' CuPy code; if it cannot run here, falls back to the scipy port and says so."""
    M = dict(OURS)
    M["TransCLIP"] = lambda Z, y, T: transclip(Z, y, T, tc_repo)
    if zlap_impl == "official":
        try:
            rng = np.random.default_rng(0); Zt = L.unit(rng.normal(size=(60, 16))); Tt = L.unit(rng.normal(size=(3, 16)))
            zlap(Zt, Tt, zl_repo)
        except Exception as e:
            print("official ZLaP could not run here (", repr(e)[:200], ") -> using the scipy port"); zlap_impl = "port"
    M["ZLaP"] = (lambda Z, y, T: zlap(Z, T, zl_repo)) if zlap_impl == "official" else (lambda Z, y, T: zlap_port(Z, T))
    global ZLAP_IMPL; ZLAP_IMPL = zlap_impl; print("ZLaP implementation:", zlap_impl)
    return M

def run_one(name, bundle, text_cache, out_path, M, keep_balanced_preds=True):
    res = json.load(open(out_path)) if os.path.exists(out_path) else {"run": name, "top1": {}, "seconds": {}}
    Z, y, names, anchors = load_run(bundle, text_cache); res["zlap_impl"] = ZLAP_IMPL; res["anchors"] = list(anchors)
    subsets = [("balanced", None)] + [(f"IR{ir}", s) for ir in IRS for s in SEEDS]
    preds = {}
    for a, T in anchors.items():
        for m, f in M.items():
            key = f"{m} | {a}"
            if key in res["top1"]: continue
            acc = {}; t0 = time.time()
            for cond, seed in subsets:
                ix = np.arange(len(y)) if seed is None else L.skew_idx(y, int(cond[2:]), seed)
                p = f(Z[ix], y[ix], T); acc.setdefault(cond, []).append(100 * float(np.mean(p == y[ix])))
                if seed is None and keep_balanced_preds: preds[key] = (p == y[ix])
            res["top1"][key] = {c: float(np.mean(v)) for c, v in acc.items()}; res["seconds"][key] = time.time() - t0
            json.dump(res, open(out_path, "w"), indent=1)
            r = res["top1"][key]; print(f"  {name:20s} {key:22s} bal {r['balanced']:6.2f}  IR10 {r['IR10']:6.2f}  IR50 {r['IR50']:6.2f}   {res['seconds'][key]:.0f}s", flush=True)
    if preds: np.savez_compressed(out_path.replace(".json", "_balanced_correct.npz"), **{k.replace(" | ", "__"): v for k, v in preds.items()})
    return res

# numbers in the paper (Table 2), used to confirm this notebook reproduces them
EXPECT = {("esc50", "imagebind"): {"cosine | T": 68.30, "EM-WO+P | T": 70.50, "WO+P | T": 79.10, "EM-WO+P | T+D": 75.90},
          ("us8k", "imagebind"): {"cosine | T": 48.91, "EM-WO+P | T": 51.73, "WO+P | T": 53.73, "EM-WO+P | T+D": 67.37},
          ("vggsound", "imagebind"): {"cosine | T": 27.59, "EM-WO+P | T": 30.03, "WO+P | T": 32.24, "EM-WO+P | T+D": 31.42},
          ("us8k", "languagebind"): {"cosine | T": 80.18, "EM-WO+P | T": 83.20, "WO+P | T": 84.24, "EM-WO+P | T+D": 86.39}}


In [ ]:
%%writefile v17_vision.py
"""v17_vision.py - the Table-2 imbalance protocol on image and video (ImageNet-1k, Places365, Kinetics-400 subset),
with TransCLIP and ZLaP (authors' code) next to cosine, WO+P and EM-WO+P. Templates only (no LLM descriptions exist
for these class sets). Built on v16_baselines.py; same subsamples (llm_lib.skew_idx, seeds 0-2) and metric.

Memory-only changes so that TransCLIP fits a 15 GB T4 on 50k images x 1000 classes (results are unchanged):
* its two chunked loops (Gaussian likelihood, covariance update) use a chunk size set per dataset instead of a fixed 2500
  (2500 is kept whenever it fits, i.e. on every audio set);
* for N > 20,000 its 3-NN affinity is computed in row blocks instead of one dense N x N matrix (same neighbours, same weights)."""
import sys, numpy as np, torch
import v16_baselines as B

BIG_N = 20000
_ORIG_BAM = None

def _bam_chunked(query_features, support_features, num_samples, n_neighbors=3, block=4096):
    """Zero-shot case of TransCLIP's build_affinity_matrix, computed in row blocks."""
    assert support_features is None
    q = query_features; N = q.size(0); dev = q.device; cols, vals = [], []
    for s in range(0, N, block):
        A = q[s:s + block] @ q.T
        top = A.topk(n_neighbors + 1, -1, largest=True).indices[:, 1:]      # drop the first hit (the sample itself), as theirs
        cols.append(top); vals.append(A.gather(1, top)); del A
    cols = torch.cat(cols).flatten().cpu(); vals = torch.cat(vals).flatten()
    rows = torch.arange(N).unsqueeze(1).repeat(1, n_neighbors).flatten()
    return torch.sparse_coo_tensor(torch.stack([rows, cols]).to(dev), vals.to(dev), size=(N, N), device=dev)

def transclip_big(Z, y, T, repo, force_chunked=False):
    global _ORIG_BAM
    B._transclip_solver(repo)
    TC = sys.modules["TransCLIP_solver.TransCLIP"]; TCU = sys.modules["TransCLIP_solver.TransCLIP_utils"]
    assert hasattr(TCU, "CHUNK"), "run the clone cell first (it patches TransCLIP's fixed chunk size)"
    if _ORIG_BAM is None: _ORIG_BAM = TC.build_affinity_matrix
    K, d = T.shape
    TCU.CHUNK = int(max(32, min(2500, 1.0e9 / (K * d * 4))))               # ~1 GB per chunk of (chunk x K x d) floats
    TC.build_affinity_matrix = _bam_chunked if (force_chunked or len(Z) > BIG_N) else _ORIG_BAM
    return B.transclip(Z, y, T, repo)

def methods(tc_repo, zl_repo, zlap_impl="official"):
    M = B.methods(tc_repo, zl_repo, zlap_impl)
    M["TransCLIP"] = lambda Z, y, T: transclip_big(Z, y, T, tc_repo)
    return M

# Table 1 of the paper (balanced; cosine re-run and WO+P), used to confirm the bundles are the same ones
EXPECT = {("imagenet", "imagebind"): {"cosine | T": 77.53, "WO+P | T": 80.41}, ("places365", "imagebind"): {"cosine | T": 46.08, "WO+P | T": 47.91},
          ("kinetics400", "imagebind"): {"cosine | T": 52.88, "WO+P | T": 60.23}, ("imagenet", "languagebind"): {"cosine | T": 78.73, "WO+P | T": 80.79},
          ("places365", "languagebind"): {"cosine | T": 44.75, "WO+P | T": 47.15}, ("kinetics400", "languagebind"): {"cosine | T": 63.68, "WO+P | T": 66.98}}

import os, json, time
import llm_lib as L

def run_one(name, bundle, out_path, M, seeds=(0, 1, 2), irs=(10, 50)):
    """Like v16_baselines.run_one, but saves after every single subsample, so a Colab disconnect loses at most one fit."""
    res = json.load(open(out_path)) if os.path.exists(out_path) else {"run": name, "parts": {}, "seconds": {}}
    Z, y, names, anchors = B.load_run(bundle); T = anchors["T"]
    res.update({"zlap_impl": B.ZLAP_IMPL, "N": int(len(y)), "C": int(T.shape[0])})
    subsets = [("balanced", None)] + [(f"IR{ir}", s) for ir in irs for s in seeds]
    print(f"{name}: N={len(y):,} C={T.shape[0]}  smallest class at 50x keeps {max(1, int(round(np.bincount(y).min() / 50)))} sample(s)", flush=True)
    for m, f in M.items():
        key = f"{m} | T"; P = res["parts"].setdefault(key, {})
        for cond, seed in subsets:
            tag = f"{cond}|{seed}"
            if tag in P: continue
            ix = np.arange(len(y)) if seed is None else L.skew_idx(y, int(cond[2:]), seed)
            t0 = time.time(); p = f(Z[ix], y[ix], T); P[tag] = 100 * float(np.mean(p == y[ix]))
            res["seconds"][key] = res["seconds"].get(key, 0) + time.time() - t0
            if seed is None: np.save(out_path.replace(".json", f"_{m}_balanced_correct.npy"), p == y[ix])
            json.dump(res, open(out_path, "w"), indent=1)
        agg = {c: float(np.mean([v for t, v in P.items() if t.split("|")[0] == c])) for c in ("balanced", "IR10", "IR50")}
        print(f"  {name:24s} {key:16s} bal {agg['balanced']:6.2f}  IR10 {agg['IR10']:6.2f}  IR50 {agg['IR50']:6.2f}   {res['seconds'][key]:.0f}s", flush=True)
    return res

def summarize(paths):
    rows = []
    for p in paths:
        r = json.load(open(p)); ds, enc = r["run"].split("/")
        for key, P in r["parts"].items():
            m = key.split(" | ")[0]
            row = {"dataset": ds, "encoder": enc, "method": m, "anchors": "T"}
            for c in ("balanced", "IR10", "IR50"):
                v = [x for t, x in P.items() if t.split("|")[0] == c]; row[c] = round(float(np.mean(v)), 2) if v else None
                if c != "balanced": row[c + "_sd"] = round(float(np.std(v)), 2) if v else None
            row["seconds"] = round(r["seconds"].get(key, 0), 1); row["zlap_impl"] = r.get("zlap_impl"); rows.append(row)
    return rows


In [ ]:
# ---- main run (resumable after every subsample: results/v17_<ds>_<enc>.json)
import importlib, v16_baselines as B, v17_vision as V
importlib.reload(B); importlib.reload(V)
M = V.methods("/content/_TransCLIP", "/content/_ZLaP")
for r in RUNS:
    print("==", r["ds"], r["enc"])
    V.run_one(f"{r['ds']}/{r['enc']}", r["bundle"], f"results/v17_{r['ds']}_{r['enc']}.json", M)
    import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- results: per run, mean over the runs, check against Table 1, and audio + vision together
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
df = pd.DataFrame(V.summarize(sorted(glob.glob("results/v17_*.json"))))
df.to_csv("results/v17_imbalance_vision.csv", index=False)
order = ["cosine", "TransCLIP", "ZLaP", "WO+P", "EM-WO+P"]
print(df.pivot_table(index="method", columns=["dataset", "encoder"], values=["balanced", "IR10", "IR50"]).reindex(order).round(2).to_string())
mean = df.groupby("method")[["balanced", "IR10", "IR50"]].mean().reindex(order).round(2)
mean["runs"] = df.groupby("method").size(); print("\n=== mean over the image/video runs ===\n", mean.to_string())
print("\n=== same bundles as Table 1? (balanced top-1) ===")
for (ds, enc), exp in V.EXPECT.items():
    for key, val in exp.items():
        h = df[(df.dataset == ds) & (df.encoder == enc) & (df.method == key.split(" | ")[0])]
        if len(h): print(f"{ds:12s} {enc:12s} {key:12s} paper {val:6.2f}  here {h.balanced.iloc[0]:6.2f}  {'OK' if abs(h.balanced.iloc[0] - val) < 0.06 else 'DIFFERENT'}")
if os.path.exists("results/v16_baselines.csv"):                            # audio runs from v16, templates only
    a = pd.read_csv("results/v16_baselines.csv"); a = a[(a.anchors == "T") & a.dataset.isin(["esc50", "us8k", "vggsound"])]
    both = pd.concat([a[["dataset", "encoder", "method", "balanced", "IR10", "IR50"]], df[["dataset", "encoder", "method", "balanced", "IR10", "IR50"]]])
    m2 = both.groupby("method")[["balanced", "IR10", "IR50"]].mean().reindex(order).round(2); m2["runs"] = both.groupby("method").size()
    print("\n=== audio (v16) + image/video (v17), templates only ===\n", m2.to_string())
print("\nsaved results/v17_imbalance_vision.csv")

**Send back:** `MyDrive/ortho-v7/results/v17_imbalance_vision.csv`. The `v17_*.json` files are optional; they hold every individual subsample. The `_balanced_correct.npy` files allow McNemar tests.